# Project 2

셀을 차례대로 실행하면 됩니다.

In [ ]:
'''
1. Clone or Pull Repository
코드는 github에 올려서 사용하였습니다.
'''

%cd /content/

import os
REPO_URL = "https://github.com/cyunchaeskku/AILab_Project2_submission.git"
REPO_NAME = "AILab_Project2_submission"

if os.path.exists(REPO_NAME):
    print(f"Pulling latest changes for {REPO_NAME}...")
    !cd {REPO_NAME} && git pull
else:
    print(f"Cloning {REPO_NAME}...")
    !git clone {REPO_URL}

In [ ]:
'''
2. Install dependencies
requirements.txt를 설치합니다.
'''
!pip install gdown
%cd AILab_Project2_submission
!pip install -r requirements.txt

In [ ]:
'''
3. Mount Google Drive for Checkpoints
google drive를 mount합니다.
체크포인트를 google drive에 저장하기 위해서 mount는 필수입니다.
'''
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
DRIVE_PROJECT = Path("/content/drive/MyDrive/OpenAILab/project2")
!ls -lh /content/drive/MyDrive/OpenAILab/project2

In [ ]:
'''
4. Download Train/Validation Datasets
데이터셋을 다운로드 합니다.
'''
from pathlib import Path
import shutil
import subprocess

DATA_DIR = Path("data")
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/OpenAILab/project2/train_val_data")
DATA_DIR.mkdir(exist_ok=True)
DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)

files = {
    # Training data only. Use these for model training.
    "train_50k_256.zip": "1sz8s31qCWwuCTWT_DQK2_yoQ6pjtUZvG",
    "train_50k_512.zip": "1R1JBbYGfeOPZedKHjuJ5VZavVdjXuJl-",
    "train_50k_1024.zip": "1qFJN3QOmM8kjwgWKztHPIqTvZFuZnZLd",

    # Validation data only. Do not use these for training.
    "valid_10k_256.zip": "1yEA26_FEgHwPur2jZCr0lmONesA6TyI2",
    "valid_10k_512.zip": "1hLDAyFgTZA2KRYSdJvPLhHJ3K0AgWfpE",
    "valid_10k_1024.zip": "1V0HfqDxUytt-wOiqyerXYC1-61EE8HqZ",
}

for filename, file_id in files.items():
    out = DATA_DIR / filename
    cached = DRIVE_DATA_DIR / filename

    if out.exists() and out.stat().st_size > 0:
        print(f"skip local {out} ({out.stat().st_size / 1024**3:.2f} GB)")
        if not cached.exists() or cached.stat().st_size == 0:
            print(f"cache to Drive {cached}")
            shutil.copy2(out, cached)
        continue

    if cached.exists() and cached.stat().st_size > 0:
        print(f"copy from Drive cache {cached} -> {out}")
        shutil.copy2(cached, out)
        continue

    print(f"download {out}")
    url = f"https://drive.google.com/uc?id={file_id}"
    result = subprocess.run(["gdown", url, "-O", str(out)], check=False)
    if result.returncode != 0 or not out.exists() or out.stat().st_size == 0:
        print(f"download failed: {filename}")
        if out.exists() and out.stat().st_size == 0:
            out.unlink()
        continue

    print(f"cache to Drive {cached}")
    shutil.copy2(out, cached)

!ls -lh data
!ls -lh /content/drive/MyDrive/OpenAILab/project2/train_val_data


In [ ]:
'''
5. Verify Baseline Sample
baseline checkpoint로부터 샘플을 생성하여 체크할 수 있습니다.
'''
!python generate.py --ckpt ckpt/ffhq256_baseline.pt --out sample_256.png --n 16 --seed 42

In [ ]:
'''
6. Login to Weights & Biases
wandb에 로그인합니다. (api key 직접 입력 필요)
'''
import getpass
import os
import wandb

if not os.environ.get("WANDB_API_KEY"):
    os.environ["WANDB_API_KEY"] = getpass.getpass("WANDB_API_KEY: ")

wandb.login(key=os.environ["WANDB_API_KEY"])


In [ ]:
'''
7. Clean GPU Processes
OOM 에러를 방지하기 위해 혹시나 실행되고 있는 다른 GPU 프로세스를 종료합니다.
'''
import gc
import os
import signal
import subprocess

print("GPU processes before cleanup:")
!nvidia-smi

current_pid = os.getpid()
cmd = [
    "nvidia-smi",
    "--query-compute-apps=pid,process_name,used_memory",
    "--format=csv,noheader,nounits",
]
result = subprocess.run(cmd, capture_output=True, text=True)

for line in result.stdout.strip().splitlines():
    if not line.strip():
        continue
    pid_text, name, mem_text = [part.strip() for part in line.split(",", 2)]
    pid = int(pid_text)
    if pid == current_pid:
        print(f"keep current kernel pid={pid} mem={mem_text}MiB")
        continue
    print(f"kill gpu process pid={pid} name={name} mem={mem_text}MiB")
    os.kill(pid, signal.SIGTERM)

gc.collect()
try:
    import torch
    torch.cuda.empty|_cache()
except Exception as exc:
    print(f"torch cache cleanup skipped: {exc}")

print("GPU processes after cleanup:")
!nvidia-smi


In [ ]:
'''
8. Start True Progressive Training from Professor 256 Baseline
**학습에는 A100을 사용하였습니다**
--auto-resume을 사용해 체크포인트 파일 중 가장 최근의 체크포인트부터 자동으로 resume합니다.
첫 실행이 아닌 경우 --new-wandb-run은 제거하거나 주석처리합니다.
'''
!python train.py \
  --config configs/progressive_1024_a100.yaml \
  --init-from /ckpt/ffhq256_baseline.pt \ 
  --auto-resume \
  --new-wandb-run

In [ ]:
'''
9. Generate Images
seed_number를 바꾸어 이미지를 생성할 수 있습니다.
'''

seed_number = 42

!python generate_512_upsample.py \
    --ckpt ckpt/model.pt \
    --psi 0.75 \
    --out samples/sample_{seed_number}.png --n 24 --seed {seed_number}

In [ ]:
'''
10. Export Progressive Generator to ONNX
저장된 체크포인트를 .onnx로 만듭니다. (체크포인트를 model.pt로 저장하여 ckpt/에 넣어두었음)
과제 진행 중에는 google drive의 체크포인트(ex. /content/drive/MyDrive/.../ckpt_00xxx.pt)를 가져와 onnx로 만드는 방식으로 진행했습니다.
기존 export_onnx.py를 일부 수정해 truncation을 추가한 export_onnx_truncation.py를 만들었습니다.
'''
!python export_onnx_truncated.py \
--psi 0.75 \
--ckpt ckpt/model.pt \
--out submission.onnx